In [1]:
%cd /content
!ls

/content
cctv_transforms.py  engine.py	 predict.py   train.py
config.py	    evaluate.py  sample_data


In [2]:
!apt-get install -y unrar
!pip install -q onnx onnxruntime

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:7.0.7-1build1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.2 MB/s eta 0:00:00


In [6]:
!python download_data.py

     651.3 / 651.3 MB
Extracting ...
Extracting RAR archive: smokingVSnotsmoking.rar ...

UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal


Extracting from /content/data/raw/smokingVSnotsmoking.rar

Creating    /content/data/raw/smokingVSnotsmoking/dataset             OK
Creating    /content/data/raw/smokingVSnotsmoking/dataset/testing_data  OK
Extracting  /content/data/raw/smokingVSnotsmoking/dataset/testing_data/abc001.jpg       0%  OK 
Extracting  /content/data/raw/smokingVSnotsmoking/dataset/testing_data/abc002.jpg       0%  OK 
Extracting  /content/data/raw/smokingVSnotsmoking/dataset/testing_data/abc003.jpg       0%  OK 
Extracting  /content/data/raw/smokingVSnotsmoking/dataset/testing_data/abc004.jpg       0%  OK 
Extracting  /content/data/raw/smokingVSnotsmoking/dataset/testing_data/abc005.jpg       0%  OK 
Extracting  /content/data/raw/smokingVSnotsmoking/dataset/testing_data/abc006.jpg       0%  OK 
Extra

In [7]:
!python audit_data.py

training_data {'images': 1610, 'notsmoking': 805, 'smoking': 805, 'long_side_min': 200, 'long_side_median': 956, 'long_side_max': 7057, 'share_long_side_lt_320': 0.019254658385093167}
validation_data {'images': 400, 'notsmoking': 200, 'smoking': 200, 'long_side_min': 240, 'long_side_median': 900, 'long_side_max': 7360, 'share_long_side_lt_320': 0.0275}

Train images that are near-duplicates of validation images: 24
Near-duplicates with DIFFERENT labels (label noise): 0
Excluded from training: 24 -> /content/outputs/exclude.txt


In [8]:
!python preview_cctv.py

saved /content/outputs/preview_cctv_0.png
saved /content/outputs/preview_cctv_1.png


In [17]:
!python train.py

device: cuda | AMP: True
Config(arch='mobilenet_v3_large', weights='v2', dropout=0.3, img_size=224, pad_value=114, max_side=640, epochs=100, batch_size=64, lr=0.001, weight_decay=0.01, warmup_epochs=2, sched_factor=0.5, sched_patience=2, min_lr=0.00025, es_patience=5, es_min_delta=0.0001, cctv_prob=0.8, num_workers=2, seed=42, leak_hamming=4)
train: 1586 images (783 smoking) | val: 400 images (200 smoking)
  pre-rendering 400 eval images (cctv=True) ...
Epoch   1/100 | lr 1.00e-03 | train loss 0.4945 acc 0.755 | val loss 0.4154 acc 0.818 f1 0.808 auc 0.897 | 43s *best* [warm-up]
Epoch   2/100 | lr 1.00e-03 | train loss 0.3922 acc 0.831 | val loss 0.3794 acc 0.853 f1 0.847 auc 0.913 | 41s *best* [warm-up]
Epoch   3/100 | lr 1.00e-03 | train loss 0.4914 acc 0.763 | val loss 0.8529 acc 0.698 f1 0.578 auc 0.916 | 43s [patience 1/5]
Epoch   4/100 | lr 1.00e-03 | train loss 0.3493 acc 0.850 | val loss 0.6835 acc 0.800 f1 0.788 auc 0.898 | 42s [patience 2/5]
Epoch   5/100 | lr 1.00e-03 | trai

In [13]:
%%writefile train.py
import argparse
import csv
import random
import time
import matplotlib
import numpy as np
import torch
import torch.nn as nn

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from config import CKPT_DIR, OUT_DIR, Config
from dataset import get_loaders
from engine import (EarlyStopping, compute_metrics, predict_loader, train_one_epoch)
from model import build_model, param_groups, set_backbone_trainable

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def save_history(history):
    if not history:
        return
    with open(OUT_DIR / "history.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        w.writeheader()
        w.writerows(history)

def plot_history(history):
    if not history:
        return
    ep = [h["epoch"] for h in history]
    for name, keys, ylabel in [("loss.png", ("train_loss", "val_loss"), "loss"),
                               ("accuracy.png", ("train_acc", "val_acc"), "accuracy"),
                               ("lr.png", ("lr",), "learning rate")]:
        plt.figure(figsize=(6, 4))
        for k in keys:
            plt.plot(ep, [h[k] for h in history], marker="o", ms=3, label=k)
        plt.xlabel("epoch")
        plt.ylabel(ylabel)
        if name == "lr.png":
            plt.yscale("log")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.tight_layout()
        plt.savefig(OUT_DIR / name, dpi=120)
        plt.close()

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", type=str, default=None,
                    help="Model architecture: efficientnet_b0, efficientnet_v2_s, mobilenet_v3_large, etc.")
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--batch-size", type=int)
    ap.add_argument("--lr", type=float)
    ap.add_argument("--min-lr", type=float)
    ap.add_argument("--img-size", type=int)
    ap.add_argument("--warmup-epochs", type=int)
    ap.add_argument("--es-patience", type=int, help="Early stopping patience")
    ap.add_argument("--workers", type=int)
    ap.add_argument("--limit", type=int, help="use only N images per split (debug)")
    ap.add_argument("--smoke", action="store_true", help="tiny run to verify the pipeline")
    ap.add_argument("--resume", action="store_true")
    ap.add_argument("--no-pretrained", action="store_true")
    args = ap.parse_args()

    cfg = Config()
    for k, v in [("arch", args.arch), ("epochs", args.epochs), ("batch_size", args.batch_size),
                 ("lr", args.lr), ("min_lr", args.min_lr), ("img_size", args.img_size),
                 ("warmup_epochs", args.warmup_epochs), ("es_patience", args.es_patience),
                 ("num_workers", args.workers)]:
        if v is not None:
            setattr(cfg, k, v)
    if args.smoke:
        cfg.epochs, cfg.warmup_epochs, args.limit = 2, 1, args.limit or 64
        cfg.batch_size = min(cfg.batch_size, 16)
    assert cfg.lr > cfg.min_lr, "lr must be above min_lr or ReduceLROnPlateau can never act"

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    set_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    amp = device.type == "cuda"
    print(f"device: {device} | AMP: {amp}")
    print(cfg)

    train_loader, val_loader = get_loaders(cfg, limit=args.limit)
    model = build_model(cfg, pretrained=not args.no_pretrained).to(device)
    optimizer = torch.optim.AdamW(param_groups(model, cfg.weight_decay), lr=cfg.lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=cfg.sched_factor, patience=cfg.sched_patience,
        threshold=cfg.es_min_delta, threshold_mode="abs", min_lr=cfg.min_lr)
    es = EarlyStopping(cfg.es_patience, cfg.es_min_delta)
    scaler = torch.amp.GradScaler(device.type, enabled=amp)
    criterion = nn.BCEWithLogitsLoss()

    history, start_epoch = [], 1
    last_path, best_path = CKPT_DIR / "last.pt", CKPT_DIR / "best.pt"
    if args.resume and last_path.exists():
        ck = torch.load(last_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model"])
        optimizer.load_state_dict(ck["optimizer"])
        scheduler.load_state_dict(ck["scheduler"])
        scaler.load_state_dict(ck["scaler"])
        es.load_state_dict(ck["es"])
        history, start_epoch = ck["history"], ck["epoch"] + 1
        print(f"Resumed from epoch {ck['epoch']} (best val loss {es.best:.4f})")

    t_start = time.time()
    try:
        for epoch in range(start_epoch, cfg.epochs + 1):
            frozen = epoch <= cfg.warmup_epochs
            set_backbone_trainable(model, not frozen)
            lr = optimizer.param_groups[0]["lr"]
            t0 = time.time()

            tr = train_one_epoch(model, train_loader, optimizer, scaler, criterion,
                                 device, amp, frozen)
            va = predict_loader(model, val_loader, device, criterion)
            m = compute_metrics(va["labels"], va["probs"], 0.5)

            if not frozen:
                scheduler.step(va["loss"])
            improved = es.step(va["loss"], active=not frozen)
            if improved:
                torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": va["loss"],
                            "cfg": cfg.to_dict()}, best_path)

            history.append({"epoch": epoch, "lr": lr, "train_loss": tr["loss"],
                            "train_acc": tr["acc"], "val_loss": va["loss"], "val_acc": m["accuracy"],
                            "val_f1": m["f1"], "val_auc": m["auc"], "seconds": time.time() - t0})
            save_history(history)
            torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                        "es": es.state_dict(), "epoch": epoch, "history": history,
                        "cfg": cfg.to_dict()}, last_path)

            new_lr = optimizer.param_groups[0]["lr"]
            print(f"Epoch {epoch:3d}/{cfg.epochs} | lr {lr:.2e} | train loss {tr['loss']:.4f} "
                  f"acc {tr['acc']:.3f} | val loss {va['loss']:.4f} acc {m['accuracy']:.3f} "
                  f"f1 {m['f1']:.3f} auc {m['auc']:.3f} | {time.time() - t0:.0f}s"
                  f"{' *best*' if improved else ''}"
                  f"{' [warm-up]' if frozen else f' [patience {es.counter}/{cfg.es_patience}]'}")
            if new_lr < lr:
                print(f"    ReduceLROnPlateau: lr {lr:.2e} -> {new_lr:.2e}")
            if es.should_stop:
                print(f"Early stopping at epoch {epoch} (no val-loss improvement for "
                      f"{cfg.es_patience} epochs).")
                break
    except KeyboardInterrupt:
        print("Interrupted - last.pt is saved, use --resume to continue.")

    plot_history(history)
    if best_path.exists():
        ck = torch.load(best_path, map_location="cpu", weights_only=False)
        print(f"Best epoch {ck['epoch']} | val loss {ck['val_loss']:.4f} | {best_path}")
    print(f"Total time: {(time.time() - t_start) / 60:.1f} min")

if __name__ == "__main__":
    main()


Overwriting train.py


In [14]:
!python train.py --arch efficientnet_b0

device: cuda | AMP: True
Config(arch='efficientnet_b0', weights='v2', dropout=0.3, img_size=224, pad_value=114, max_side=640, epochs=100, batch_size=64, lr=0.001, weight_decay=0.01, warmup_epochs=2, sched_factor=0.5, sched_patience=2, min_lr=0.00025, es_patience=5, es_min_delta=0.0001, cctv_prob=0.8, num_workers=2, seed=42, leak_hamming=4)
train: 1586 images (783 smoking) | val: 400 images (200 smoking)
  pre-rendering 400 eval images (cctv=True) ...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 146MB/s]
Epoch   1/100 | lr 1.00e-03 | train loss 0.5812 acc 0.728 | val loss 0.4781 acc 0.820 f1 0.805 auc 0.902 | 43s *best* [warm-up]
Epoch   2/100 | lr 1.00e-03 | train loss 0.4552 acc 0.820 | val loss 0.4166 acc 0.820 f1 0.806 auc 0.909 | 42s *best* [warm-up]
Epoch   3/100 | lr 1.00e-03 | train loss 0.5262 acc 0.759 | val loss 0.5119 acc 0.

In [15]:
!python train.py --arch efficientnet_v2_s

device: cuda | AMP: True
Config(arch='efficientnet_v2_s', weights='v2', dropout=0.3, img_size=224, pad_value=114, max_side=640, epochs=100, batch_size=64, lr=0.001, weight_decay=0.01, warmup_epochs=2, sched_factor=0.5, sched_patience=2, min_lr=0.00025, es_patience=5, es_min_delta=0.0001, cctv_prob=0.8, num_workers=2, seed=42, leak_hamming=4)
train: 1586 images (783 smoking) | val: 400 images (200 smoking)
  pre-rendering 400 eval images (cctv=True) ...
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth
100% 82.7M/82.7M [00:00<00:00, 208MB/s]
Epoch   1/100 | lr 1.00e-03 | train loss 0.6225 acc 0.682 | val loss 0.5133 acc 0.840 f1 0.838 auc 0.899 | 44s *best* [warm-up]
Epoch   2/100 | lr 1.00e-03 | train loss 0.5128 acc 0.798 | val loss 0.4387 acc 0.848 f1 0.844 auc 0.908 | 43s *best* [warm-up]
Epoch   3/100 | lr 1.00e-03 | train loss 0.4749 acc 0.781 | val loss 0.4273 acc 0.858 f1 0.856 a

In [20]:
!python benchmark_latency.py \
    /content/outputs/checkpoints/efficientnet_b0_best.pt \
    /content/outputs/checkpoints/efficientnet_v2_s_best.pt \
    /content/outputs/checkpoints/mobilenet_v3_large_best.pt


 Complete Model Benchmark: GPU vs CPU
 GPU: Tesla T4 (150 runs)
 CPU: Host Processor (50 runs)

-> Testing efficientnet_b0 (efficientnet_b0_best.pt) ...
-> Testing efficientnet_v2_s (efficientnet_v2_s_best.pt) ...
-> Testing mobilenet_v3_large (mobilenet_v3_large_best.pt) ...

Model Architecture     | Params  | Size   | GPU FP16   | GPU FP32   | GPU BS=8  | CPU (BS=1)  | CPU FPS 
                       | (M)     | (MB)   | Latency    | Latency    | FPS       | Latency     |         
---------------------------------------------------------------------------------------------------------
efficientnet_b0        | 4.01 M | 15.6 M | 9.54 ms    | 8.16 ms    | 781.0     | 36.05 ms    | 27.7    
efficientnet_v2_s      | 20.18M | 77.8 M | 23.41 ms   | 18.64 ms   | 348.5     | 121.36 ms   | 8.2     
mobilenet_v3_large     | 4.20 M | 16.2 M | 8.36 ms    | 6.55 ms    | 1035.3    | 21.73 ms    | 46.0    
Note: Latency is in milliseconds per frame (lower is better). FPS is frames per second (highe